In [ ]:
!pip install roboflow

  Using cached roboflow-1.2.11-py3-none-any.whl.metadata (9.7 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached pi_heif-1.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.5 kB)
  Using cached pillow_avif_plugin-1.5.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached roboflow-1.2.11-py3-none-any.whl (89 kB)
Using cached idna-3.7-py3-none-any.whl (66 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.9 MB)
Using cached pi_heif-1.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)
Using cached pillow_avif_plugin-1.5.2-cp312-cp312-manylinux_2_28_x86_64.whl (4.2 MB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
  Attempting uninstall: openc

In [ ]:
# Install required packages
!pip install roboflow ultralytics torch torchvision

import torch
import os
import yaml
from roboflow import Roboflow
from ultralytics import YOLO
import shutil

# Initialize Roboflow and download dataset
rf = Roboflow(api_key="CRShVXHDxmBE549PwD0B")
project = rf.workspace("fire-fs3r3").project("merged-satellite-flood-images")
version = project.version(4)
dataset = version.download("yolov11")

print("Dataset downloaded successfully!")
print(f"Dataset location: {dataset.location}")

# Check the actual directory structure
def explore_directory(path, indent=0):
    """Recursively explore directory structure"""
    print(" " * indent + f"📁 {os.path.basename(path)}/")
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            explore_directory(item_path, indent + 2)
        else:
            print(" " * (indent + 2) + f"📄 {item}")

explore_directory(dataset.location)

# Find the correct data.yaml file and fix paths
def find_and_fix_data_yaml(root_path):
    """Find data.yaml and fix the paths"""
    data_yaml_path = None
    # Search for data.yaml
    for root, dirs, files in os.walk(root_path):
        if 'data.yaml' in files:
            data_yaml_path = os.path.join(root, 'data.yaml')
            print(f"Found data.yaml at: {data_yaml_path}")
            break # Found it, no need to search further

    if data_yaml_path:
        # Read the data.yaml
        with open(data_yaml_path, 'r') as f:
            data = yaml.safe_load(f)

        print("Original data.yaml content:")
        print(data)

        # Fix paths - ensure 'path' key exists and paths are correct relative to the dataset root
        base_dir = dataset.location # Use the base download location as the root

        # Add 'path' key if missing or incorrect
        data['path'] = base_dir

        # Update paths to be relative to the base_dir or absolute if needed
        for key in ['train', 'val', 'test', 'valid']:
            if key in data:
                current_path = data[key]
                if current_path and not os.path.isabs(current_path):
                    # If the path is relative, make it absolute using the data.yaml directory as context first
                    abs_path_from_yaml = os.path.join(os.path.dirname(data_yaml_path), current_path)
                    # Then make it relative to the base_dir or keep absolute if already correct
                    if os.path.commonpath([base_dir, abs_path_from_yaml]) == base_dir:
                         data[key] = os.path.relpath(abs_path_from_yaml, base_dir)
                    else:
                        data[key] = abs_path_from_yaml # Keep as absolute if it's outside the base_dir

                # Ensure 'val' and 'valid' point to the same field
                if key == 'valid' and 'val' not in data:
                     data['val'] = data['valid']
                     del data['valid']


        # Save the fixed data.yaml
        fixed_yaml_path = '/content/fixed_data.yaml'
        with open(fixed_yaml_path, 'w') as f:
            yaml.dump(data, f, default_flow_style=False)

        print("Fixed data.yaml content:")
        print(data)
        print(f"Fixed data.yaml saved to: {fixed_yaml_path}")

        return fixed_yaml_path, base_dir
    else:
        print("data.yaml not found in the downloaded dataset structure.")
        return None, None


# Find and fix the data.yaml
fixed_yaml_path, dataset_base_dir = find_and_fix_data_yaml(dataset.location)

if fixed_yaml_path is None:
     # If data.yaml was not found, we can't proceed with the standard training approach
     print("Cannot proceed with training without a valid data.yaml file.")
else:
    # Verify the paths exist
    print("\nVerifying paths...")
    with open(fixed_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Use 'val' as the validation key consistently
    keys_to_check = ['train', 'val']
    if 'test' in data_config:
        keys_to_check.append('test')

    all_paths_exist = True
    for key in keys_to_check:
        if key in data_config:
            path = os.path.join(data_config['path'], data_config[key]) if not os.path.isabs(data_config[key]) else data_config[key]
            print(f"{key} path: {path}")
            if os.path.exists(path):
                print(f"✅ {key} path exists")
                # Count images
                images_path = os.path.join(path, 'images')
                if os.path.exists(images_path):
                    num_images = len([f for f in os.listdir(images_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
                    print(f"   - {num_images} images found")
                else:
                    print(f"   - ❌ images folder not found at {images_path}")
                    all_paths_exist = False

            else:
                print(f"❌ {key} path does not exist")
                all_paths_exist = False
        else:
            print(f"❌ {key} key not found in data.yaml")
            all_paths_exist = False


    if all_paths_exist:
        # Initialize YOLOv11 model
        model = YOLO('yolo11n.pt')  # Start with nano for faster training

        # Training configuration
        training_config = {
            'data': fixed_yaml_path,  # Use the fixed data.yaml
            'epochs': 50,  # Reduced for testing
            'imgsz': 640,
            'batch': 8,  # Reduced batch size
            'lr0': 0.01,
            'device': '0' if torch.cuda.is_available() else 'cpu',
            'workers': 2,
            'patience': 10,
            'save': True,
            'name': 'yolov11_fire_flood_fixed'
        }

        print(f"\nUsing device: {training_config['device']}")
        if training_config['device'] != 'cpu':
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

        # Start training
        print("\nStarting training with fixed paths...")
        try:
            results = model.train(
                data=training_config['data'],
                epochs=training_config['epochs'],
                imgsz=training_config['imgsz'],
                batch=training_config['batch'],
                device=training_config['device'],
                workers=training_config['workers'],
                patience=training_config['patience'],
                save=training_config['save'],
                name=training_config['name']
            )
            print("Training completed successfully!")

            # Validate the model
            print("Starting validation...")
            validation_results = model.val()

            print("\nValidation Metrics:")
            print(f"mAP50: {validation_results.box.map50:.4f}")
            print(f"mAP50-95: {validation_results.box.map:.4f}")
            print(f"Precision: {validation_results.box.p:.4f}")
            print(f"Recall: {validation_results.box.r:.4f}")

            # Save the trained model
            model.export(format='pt')
            print("Model exported successfully!")

        except Exception as e:
            print(f"Training failed with error: {e}")
            print("Please review the error message and the data.yaml file.")

    else:
        print("\nTraining cannot start because one or more required data paths do not exist.")

  Using cached ultralytics-8.3.221-py3-none-any.whl.metadata (37 kB)
  Using cached ultralytics_thop-2.0.17-py3-none-any.whl.metadata (14 kB)
Using cached ultralytics-8.3.221-py3-none-any.whl (1.1 MB)
Using cached ultralytics_thop-2.0.17-py3-none-any.whl (28 kB)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Merged-Satellite-Flood-Images-4 in yolov11:: 100%|██████████| 2144/2144 [00:00<00:00, 3929.06it/s]


Dataset downloaded successfully!
Dataset location: /content/Merged-Satellite-Flood-Images-4
📁 Merged-Satellite-Flood-Images-4/
  📁 test/
    📁 images/
      📄 7485_jpg.rf.cf893055fa911fdf7c5bbc73211cd487.jpg
      📄 6996_JPG.rf.2cf986389e3e3cc08f9984e21a50fcb0.jpg
      📄 9089_JPG.rf.2a90e66e5763547caabf6323ffb2bed8.jpg
      📄 8190_JPG.rf.052c5da7983ff93646fb9f9c6e88a448.jpg
      📄 8193_JPG.rf.e95bced8dca133d5a795c9dffbfdacae.jpg
      📄 8056_JPG.rf.50ff544f02248612682b9400519440dd.jpg
      📄 9025_JPG.rf.90295b4362a6f62462e744e461e1a3a2.jpg
      📄 8386_JPG.rf.75b197f85c9704c97471611537381779.jpg
      📄 8987_JPG.rf.a6d8cd64564013170d8ad36221ef1b6b.jpg
      📄 6635_JPG.rf.4c2feab7e267e75a7e6ac9b94d81fb18.jpg
      📄 6555_JPG.rf.f0b4a9a0ca309c0b01541cd8a32457e7.jpg
      📄 7182_JPG.rf.8fdd137ec5e5dbb3f4e7d453ca05eef4.jpg
      📄 8545_JPG.rf.e55a03bcf98e7e8982117f087ee0191e.jpg
      📄 8210_JPG.rf.41e9805b46563817edcdb39252876a8a.jpg
      📄 8418_JPG.rf.c2d45281dac1d4f09641764953aadc9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Alternative approach: Manual dataset setup
import glob

def setup_dataset_manually(base_path):
    """Manually set up dataset structure"""

    # Find all image files
    train_images = glob.glob(os.path.join(base_path, '**', 'train', 'images', '*.jpg'), recursive=True)
    train_images += glob.glob(os.path.join(base_path, '**', 'train', 'images', '*.png'), recursive=True)

    val_images = glob.glob(os.path.join(base_path, '**', 'valid', 'images', '*.jpg'), recursive=True)
    val_images += glob.glob(os.path.join(base_path, '**', 'valid', 'images', '*.png'), recursive=True)

    print(f"Found {len(train_images)} training images")
    print(f"Found {len(val_images)} validation images")

    # Create a simple data.yaml
    data_content = {
        'path': base_path,
        'train': os.path.join(base_path, 'train'),
        'val': os.path.join(base_path, 'valid'),
        'nc': 2,  # Adjust based on your classes
        'names': ['fire', 'flood']  # Adjust based on your classes
    }

    manual_yaml_path = '/content/manual_data.yaml'
    with open(manual_yaml_path, 'w') as f:
        yaml.dump(data_content, f, default_flow_style=False)

    return manual_yaml_path

# Try manual setup
manual_yaml = setup_dataset_manually(dataset.location)

# Try training with manual yaml
model = YOLO('yolo11n.pt')
results = model.train(
    data=manual_yaml,
    epochs=30,
    imgsz=640,
    batch=8,
    device='0' if torch.cuda.is_available() else 'cpu'
)

Found 939 training images
Found 82 validation images
Ultralytics 8.3.221 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/manual_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Dataset path: /content/drive/MyDrive/Colab Notebooks/Flood-AI-Train/Merged Satellite Flood Images.v4i.yolov11
💾 Model save path: /content/drive/MyDrive/Colab Notebooks/Flood-AI-Train/Merged Satellite Flood Images.v4i.yolov11/trained_models

🔍 Checking train folder:
Images: /content/drive/MyDrive/Colab Notebooks/Flood-AI-Train/Merged Satellite Flood Images.v4i.yolov11/train/images -> 939
Labels: /content/drive/MyDrive/Colab Notebooks/Flood-AI-Train/Merged Satellite Flood Images